In [3]:
import os, time, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights, vgg16, VGG16_Weights
from sklearn.metrics import accuracy_score, f1_score

d:\dl_gpu_env\Lib\site-packages\PIL\Image.py:116: RuntimeWarning: The _imaging extension was built for another version of Pillow or PIL:
Core version: 13.0.0
Pillow version: 12.0.0
  warnings.warn(str(v), RuntimeWarning)


ImportError: The _imaging extension was built for another version of Pillow or PIL:
Core version: 13.0.0
Pillow version: 12.0.0

In [13]:
import sys, torch
torch.hub.set_dir("D:/torch_cache")
print("Kernel path:", sys.executable)
print("CUDA available:", torch.cuda.is_available())

Kernel path: d:\dl_gpu_env\Scripts\python.exe
CUDA available: True


In [1]:
import os, sys
os.environ["TORCH_HOME"] = "D:/torch_cache"
import torch
torch.hub.set_dir("D:/torch_cache")
assert "dl_gpu_env" in sys.executable.lower(), f"STOP! Switch kernel to D:/dl_gpu_env! Currently: {sys.executable}"
print("Verified D: drive GPU Kernel:", sys.executable, "| CUDA:", torch.cuda.get_device_name(0))

Verified D: drive GPU Kernel: d:\dl_gpu_env\Scripts\python.exe | CUDA: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
os.environ["TORCH_HOME"] = "D:/torch_cache"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR      = PROJECT_ROOT / "data" / "raw" / "images"
SPLITS_DIR   = PROJECT_ROOT / "data" / "splits"
MODELS_DIR   = PROJECT_ROOT / "models"
EMB_DIR      = PROJECT_ROOT / "artifacts" / "embeddings"
REPORTS_DIR  = PROJECT_ROOT / "reports"
EXP_CSV_PATH = REPORTS_DIR / "experiment_comparison.csv"

In [4]:
train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df   = pd.read_csv(SPLITS_DIR / "val.csv")
test_df  = pd.read_csv(SPLITS_DIR / "test.csv")
with open(SPLITS_DIR / "label_mapping.json") as f:
    mapping = json.load(f)
NUM_CLASSES = len(mapping["label_to_idx"])
idx_to_label = {int(k): v for k, v in mapping["idx_to_label"].items()}

In [5]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [6]:
class FashionRetailDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(self.img_dir / f"{row['id']}.jpg") as img:
            image = img.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label_idx"]), str(row["id"])

In [7]:
@torch.no_grad()
def extract_embeddings_fast(manifest_df, encoder_fn, desc="Extracting", batch_size=256):
    records = [(IMG_DIR / f"{r['id']}.jpg", int(r["label_idx"]), str(r["id"])) for _, r in manifest_df.iterrows()]
    def _load(args):
        with Image.open(args[0]) as im:
            return eval_transform(im.convert("RGB")), args[1], args[2]
    embs, labels = [], []
    with ThreadPoolExecutor(max_workers=8) as pool:
        for i in tqdm(range(0, len(records), batch_size), desc=desc):
            batch = list(pool.map(_load, records[i:i+batch_size]))
            imgs = torch.stack([b[0] for b in batch]).to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                feats = encoder_fn(imgs).float()
            feats = nn.functional.normalize(feats, p=2, dim=1)
            embs.append(feats.cpu().numpy())
            labels.append(np.array([b[1] for b in batch], dtype=np.int64))
    return np.vstack(embs).astype(np.float32), np.concatenate(labels)

In [8]:
def evaluate_retrieval_numpy(query_emb, query_labels, gallery_emb, gallery_labels, k=5):
    t0 = time.perf_counter()
    sim = np.matmul(query_emb, gallery_emb.T)
    top_k = np.argpartition(-sim, kth=k, axis=1)[:, :k]
    row_idx = np.arange(len(query_emb))[:, None]
    top_k = top_k[row_idx, np.argsort(-sim[row_idx, top_k], axis=1)]
    latency_ms = ((time.perf_counter() - t0) * 1000.0) / len(query_emb)

    rel = (gallery_labels[top_k] == query_labels[:, None]).astype(np.float32)
    p5 = float(np.mean(np.sum(rel, axis=1) / k))
    gal_cnt = np.maximum(np.bincount(gallery_labels, minlength=NUM_CLASSES)[query_labels], 1)
    r5 = float(np.mean(np.sum(rel, axis=1) / gal_cnt))
    prec_i = np.cumsum(rel, axis=1) / np.arange(1, k + 1)[None, :]
    map5 = float(np.mean(np.sum(prec_i * rel, axis=1) / np.minimum(gal_cnt, k)))
    return {"precision_at_5": round(p5, 4), "recall_at_5": round(r5, 4), "map_at_5": round(map5, 4),
            "query_latency_ms": round(latency_ms, 4), "index_size_mb": round(gallery_emb.nbytes / (1024**2), 2)}

In [9]:
def log_experiment_row(row_dict):
    cols = ["experiment_id","checkpoint","split","accuracy","macro_f1","precision_at_5","recall_at_5","map","query_latency_ms","index_size_mb","notes"]
    df_ex = pd.read_csv(EXP_CSV_PATH) if EXP_CSV_PATH.exists() else pd.DataFrame(columns=cols)
    df_ex = df_ex[~((df_ex["experiment_id"] == row_dict["experiment_id"]) & (df_ex["split"] == row_dict["split"]))]
    pd.concat([df_ex, pd.DataFrame([row_dict])], ignore_index=True)[cols].to_csv(EXP_CSV_PATH, index=False)

loading vgg 16 for comparison 

In [10]:
vgg_base = vgg16(weights=VGG16_Weights.DEFAULT)
vgg_encoder = nn.Sequential(
    vgg_base.features,               # Conv base up to block5_conv3 + MaxPool -> [B, 512, 7, 7]
    nn.AdaptiveAvgPool2d((1, 1)),    # Global Average Pooling -> [B, 512, 1, 1]
    nn.Flatten(start_dim=1)          # -> [B, 512]
).eval().to(DEVICE)

vgg_conv_params = sum(p.numel() for p in vgg_encoder.parameters())
print(f"VGG-16 Conv+GAP Params: {vgg_conv_params:,} (Full VGG-16 w/ FC: {sum(p.numel() for p in vgg_base.parameters()):,})")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to D:/torch_cache\hub\checkpoints\vgg16-397923af.pth
100%|██████████| 528M/528M [03:57<00:00, 2.33MB/s] 


VGG-16 Conv+GAP Params: 14,714,688 (Full VGG-16 w/ FC: 138,357,544)


In [11]:
cat_emb_vgg,  cat_lbl_vgg  = extract_embeddings_fast(train_df, vgg_encoder, desc="VGG-16 Catalog")
test_emb_vgg, test_lbl_vgg = extract_embeddings_fast(test_df,  vgg_encoder, desc="VGG-16 Test")

VGG-16 Test: 100%|██████████| 18/18 [00:49<00:00,  2.72s/it]


In [12]:
vgg_metrics = evaluate_retrieval_numpy(test_emb_vgg, test_lbl_vgg, cat_emb_vgg, cat_lbl_vgg, k=5)
log_experiment_row({
    "experiment_id": "Baseline_VGG16_PMC12367692",
    "checkpoint": "ImageNet1K_VGG16_ConvGAP",
    "split": "test",
    "accuracy": "N/A", "macro_f1": "N/A",
    "precision_at_5": vgg_metrics["precision_at_5"],
    "recall_at_5": vgg_metrics["recall_at_5"],
    "map": vgg_metrics["map_at_5"],
    "query_latency_ms": vgg_metrics["query_latency_ms"],
    "index_size_mb": vgg_metrics["index_size_mb"],
    "notes": "Reproduced Sabitha & Sundar (2025) VGG16+GAP 512-D retrieval baseline on deduplicated split."
})
print("=== VGG-16 LITERATURE BASELINE (ON SAME SPLIT) ===")
print(json.dumps(vgg_metrics, indent=2))

=== VGG-16 LITERATURE BASELINE (ON SAME SPLIT) ===
{
  "precision_at_5": 0.7617,
  "recall_at_5": 0.0085,
  "map_at_5": 0.7173,
  "query_latency_ms": 0.2361,
  "index_size_mb": 40.83
}


In [2]:
class ResNet18RetailClassifier(nn.Module):
    def __init__(self, num_classes=51, dropout_p=0.2):
        super().__init__()
        base = resnet18(weights=ResNet18_Weights.DEFAULT)
        for p in base.parameters():
            p.requires_grad = False
        # Children indices: 0:conv1, 1:bn1, 2:relu, 3:maxpool, 4:layer1, 5:layer2, 6:layer3, 7:layer4, 8:avgpool
        self.backbone = nn.Sequential(*list(base.children())[:-1], nn.Flatten(start_dim=1))
        self.classifier_head = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes)
        )

    def extract_features(self, x):
        feats = self.backbone(x)
        return nn.functional.normalize(feats, p=2, dim=1)

    def forward(self, x):
        feats = self.extract_features(x)
        logits = self.classifier_head(feats)
        return logits, feats

NameError: name 'nn' is not defined